# RK1828 Paraformer 中文语音识别 Demo（带热词）


阿里 [Paraformer](https://github.com/modelscope/FunASR)**非自回归**中文 ASR：
一次前向直接输出全部文字（不像 Whisper 逐 token 生成），实测 RTF 0.046（约 21 倍实时）。
还支持**热词增强**：给定的词汇会被优先识别（asf_scorer + 热词解码器）。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和示例音频（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 选择音频（可选热词模式） |
| 4 | 识别 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("paraformer-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/paraformer/demo_x86_64.tar.gz"),
    ("paraformer-rknn3-rk1828", "model/asf_scorer.rknn", "demos/paraformer/model/asf_scorer.rknn"),
    ("paraformer-rknn3-rk1828", "model/asf_scorer.weight", "demos/paraformer/model/asf_scorer.weight"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "demos/paraformer/model/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "demos/paraformer/model/asr_example_hotword.wav"),
    ("paraformer-rknn3-rk1828", "model/bias_encoder.bin", "demos/paraformer/model/bias_encoder.bin"),
    ("paraformer-rknn3-rk1828", "model/hotwords.txt", "demos/paraformer/model/hotwords.txt"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder.rknn", "demos/paraformer/model/paraformer_decoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder.weight", "demos/paraformer/model/paraformer_decoder.weight"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder_hotword.rknn", "demos/paraformer/model/paraformer_decoder_hotword.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder_hotword.weight", "demos/paraformer/model/paraformer_decoder_hotword.weight"),
    ("paraformer-rknn3-rk1828", "model/paraformer_encoder.rknn", "demos/paraformer/model/paraformer_encoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_encoder.weight", "demos/paraformer/model/paraformer_encoder.weight"),
    ("paraformer-rknn3-rk1828", "model/seaco_decoder.rknn", "demos/paraformer/model/seaco_decoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/seaco_decoder.weight", "demos/paraformer/model/seaco_decoder.weight"),
    ("paraformer-rknn3-rk1828", "model/seg_dict", "demos/paraformer/model/seg_dict"),
    ("paraformer-rknn3-rk1828", "model/tokens.json", "demos/paraformer/model/tokens.json"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/paraformer/demo_x86_64.tar.gz", "demos/paraformer")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 选择音频和模式

- **普通模式**：`paraformer_encoder` + `paraformer_decoder`
- **热词模式**：改用 `paraformer_decoder_hotword` + `asf_scorer`，按 `model/hotwords.txt`
  里的热词增强识别（示例热词音频：datasets/asr/asr_example_hotword.wav）

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="音频:", layout=widgets.Layout(width="60%"))
if any(o.endswith('/asr_example.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/asr_example.wav')][0]
w_hot = widgets.Checkbox(value=False, description="热词模式")
display(w_wav, w_hot)
print("选好后继续往下运行 ▼")

先听一下这段音频：

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. 识别

In [ ]:
import os

DEMO = "demos/paraformer"
wav = os.path.abspath(w_wav.value)
if w_hot.value:
    print(open(os.path.join(DEMO, "model/hotwords.txt")).read())
    run_demo(["./rknn_paraformer_demo",
              "--encoder_path", "model/paraformer_encoder.rknn",
              "--decoder_path", "model/paraformer_decoder_hotword.rknn",
              "--tokens_path", "model/tokens.json",
              "--audio_path", wav,
              "--seaco_path", "model/seaco_decoder.rknn",
              "--hotwords_path", "model/hotwords.txt",
              "--asf_path", "model/asf_scorer.rknn",
              "--asf_nfilter", "50"], DEMO)
else:
    run_demo(["./rknn_paraformer_demo",
              "--encoder_path", "model/paraformer_encoder.rknn",
              "--decoder_path", "model/paraformer_decoder.rknn",
              "--tokens_path", "model/tokens.json",
              "--audio_path", wav], DEMO)

## 小结

- 非自回归 = 并行输出全部 token，比自回归 ASR 快一个数量级，适合实时场景。
- 热词机制适合行业术语、人名地名等长尾词的识别纠偏。